# Phase 0: 현재 모델의 진짜 실력 측정

## Devil's Advocate 핵심 질문
> "AUC 0.7347이라는 숫자를 믿을 수 있는가?  
> random_state=42 하나의 분할에 의존한 결과가 아닌가?"

---

### 실험 E0-1: Stratified Repeated K-Fold로 신뢰구간 측정
### 실험 E0-2: Target 정의 민감도 분석

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RepeatedStratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report
)
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('Libraries loaded.')

## 1. 데이터 로드 및 현재 파이프라인 재현

In [ ]:
# ============================================================
# 경로를 환경에 맞게 수정하세요
# ============================================================
GOLDEN_PATH = r'C:\Users\cozy1\Documents\276_Scoring_Model\03_flowscore_ml_ver2\outputs\AUS_v2_Golden_Training_Set.csv'
TRAIN_PATH  = r'C:\Users\cozy1\Documents\276_Scoring_Model\03_flowscore_ml_ver2\outputs\AUS_v2_Ready_To_Train.csv'

# 데이터 로드 시도 (경로가 안 맞으면 현재 디렉토리에서 탐색)
import os
for path in [TRAIN_PATH, GOLDEN_PATH]:
    if os.path.exists(path):
        df = pd.read_csv(path, dtype={'COMPANY_ID_NORM': str})
        print(f'Loaded: {path}')
        break
else:
    # 현재 디렉토리 기준 탐색
    for f in ['AUS_v2_Ready_To_Train.csv', 'AUS_v2_Golden_Training_Set.csv']:
        if os.path.exists(f):
            df = pd.read_csv(f, dtype={'COMPANY_ID_NORM': str})
            print(f'Loaded: {f}')
            break
    else:
        raise FileNotFoundError('데이터 파일을 찾을 수 없습니다. 경로를 확인하세요.')

print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["TARGET_Y"].value_counts()}')

In [ ]:
# ============================================================
# 현재 v4.9 파이프라인 정확히 재현
# ============================================================

# 1. DEBT_RATIO 타입 보정
df['DEBT_RATIO'] = pd.to_numeric(df['DEBT_RATIO'], errors='coerce')

# 2. 파생 피처 생성 (feature_engineering.ipynb 로직 재현)
cols_to_fix = ['DEBT_RATIO', 'SALES_REVENUE', 'LINKED_PARTNERS', 'CASH_RATIO', 'OPERATING_MARGIN', 'EMPLOYEE_COUNT']
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

df['FE_LOG_REVENUE'] = np.log1p(df['SALES_REVENUE'])
df['FE_LOG_EMPLOYEE'] = np.log1p(df['EMPLOYEE_COUNT'])
df['FE_NET_DEPENDENCY'] = df['SALES_REVENUE'] / (df['LINKED_PARTNERS'] + 1)
df['FE_LIQUIDITY_STRESS'] = (df['DEBT_RATIO'] + 1) / (df['CASH_RATIO'] + 1)
df['FE_PROFIT_EFFICIENCY'] = (df['SALES_REVENUE'] * (df['OPERATING_MARGIN']/100)) / (df['EMPLOYEE_COUNT'] + 1)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# 3. Leakage 제거 + ID 제거
leakage_cols = ['MORATORIUM_COUNT', 'MORATORIUM_OVERDUE_AMOUNT', 
                'ACCOUNT_SUSPENSION_COUNT', 'CARD_ACCOUNT_COUNT',
                'NEGATIVE_COMMENT_COUNT']  # v4.6에서 제거됨
exclude_cols = ['COMPANY_ID', 'COMPANY_ID_NORM', 'TARGET_Y', 'TMP_Y'] + leakage_cols
features = [c for c in df.columns if c not in exclude_cols and c in df.select_dtypes(include=[np.number]).columns]

X = df[features]
y = df['TARGET_Y']

print(f'Features: {len(features)}')
print(f'Target: {y.sum()} positives / {len(y)} total ({y.mean()*100:.2f}%)')

---
## 실험 E0-1: 현재 모델의 진짜 AUC는?

### 가설
- random_state=42의 단일 분할은 운이 좋았을 수 있다
- 5-Fold CV x 3회 반복으로 15번 측정하면 진짜 실력이 드러난다

In [ ]:
# ============================================================
# E0-1: 기존 single split vs Repeated Stratified K-Fold
# ============================================================

# --- A. 기존 방식 재현 (single split, rs=42) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_v49 = RandomForestClassifier(
    n_estimators=700, max_depth=None, min_samples_split=10,
    min_samples_leaf=1, max_features='log2',
    class_weight='balanced_subsample', random_state=42
)
rf_v49.fit(X_train, y_train)
single_auc = roc_auc_score(y_test, rf_v49.predict_proba(X_test)[:, 1])
print(f'[Single Split rs=42] AUC: {single_auc:.4f}')

# --- B. 다른 random_state로 얼마나 흔들리는지 ---
print('\n[Random State Sensitivity Test]')
rs_aucs = []
for rs in [0, 7, 13, 42, 99, 123, 256, 314, 512, 999]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=y)
    rf_tmp = RandomForestClassifier(
        n_estimators=700, max_depth=None, min_samples_split=10,
        class_weight='balanced_subsample', random_state=42
    )
    rf_tmp.fit(X_tr, y_tr)
    auc_tmp = roc_auc_score(y_te, rf_tmp.predict_proba(X_te)[:, 1])
    rs_aucs.append(auc_tmp)
    print(f'  rs={rs:>3d}: AUC={auc_tmp:.4f}')

print(f'\n  Mean: {np.mean(rs_aucs):.4f} | Std: {np.std(rs_aucs):.4f}')
print(f'  Range: [{np.min(rs_aucs):.4f}, {np.max(rs_aucs):.4f}]')
print(f'\n  --> 결론: std > 0.05이면 단일 분할은 신뢰 불가')

In [ ]:
# --- C. Stratified 5-Fold CV x 3 Repeats (공식 측정) ---
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

cv_aucs = cross_val_score(
    rf_v49, X, y, cv=rskf, scoring='roc_auc', n_jobs=-1
)

print('=' * 60)
print('[E0-1 결과] Stratified 5-Fold CV x 3 Repeats')
print('=' * 60)
print(f'AUC scores (15 folds): {np.round(cv_aucs, 4)}')
print(f'\nMean AUC:  {cv_aucs.mean():.4f}')
print(f'Std AUC:   {cv_aucs.std():.4f}')
print(f'95% CI:    [{cv_aucs.mean() - 1.96*cv_aucs.std():.4f}, {cv_aucs.mean() + 1.96*cv_aucs.std():.4f}]')
print(f'Min:       {cv_aucs.min():.4f}')
print(f'Max:       {cv_aucs.max():.4f}')
print(f'\nSingle Split (rs=42): {single_auc:.4f}')
print(f'차이: {single_auc - cv_aucs.mean():+.4f}')
print('=' * 60)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC 분포
axes[0].hist(cv_aucs, bins=10, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(single_auc, color='red', linestyle='--', linewidth=2, label=f'Single Split (rs=42): {single_auc:.4f}')
axes[0].axvline(cv_aucs.mean(), color='green', linestyle='-', linewidth=2, label=f'CV Mean: {cv_aucs.mean():.4f}')
axes[0].set_xlabel('AUC')
axes[0].set_ylabel('Count')
axes[0].set_title('E0-1: AUC 분포 (15 folds)')
axes[0].legend()

# RS sensitivity
axes[1].bar(range(10), rs_aucs, color='coral', edgecolor='black')
axes[1].axhline(np.mean(rs_aucs), color='green', linestyle='--', label=f'Mean: {np.mean(rs_aucs):.4f}')
axes[1].set_xlabel('Random State Index')
axes[1].set_ylabel('AUC')
axes[1].set_title('E0-1: Random State별 AUC 변동')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 실험 E0-2: Target 정의 민감도 분석

### Devil's Advocate 질문
> "57건이라는 양성 라벨의 정의를 바꾸면 모델 성능이 어떻게 변하는가?"  
> "정의가 너무 넓으면 노이즈가, 너무 좁으면 학습 부족이 발생한다"

In [ ]:
# ============================================================
# E0-2: Target 정의에 따른 AUC 변화
# ============================================================

# 원본 Golden Set에서 Y값 구성요소 확인
# 현재 TARGET_Y는 OVERDUE(11) + MORATORIUM_RISK(46) = 57

# Moratorium 관련 컬럼으로 세분화 (leakage_cols이지만 Target 정의에만 사용)
df_raw = pd.read_csv(GOLDEN_PATH if os.path.exists(GOLDEN_PATH) else 'AUS_v2_Golden_Training_Set.csv',
                     dtype={'COMPANY_ID_NORM': str})

# MORATORIUM_OVERDUE_AMOUNT, ACCOUNT_SUSPENSION_COUNT 확인
mora_cols = ['MORATORIUM_COUNT', 'MORATORIUM_OVERDUE_AMOUNT', 'ACCOUNT_SUSPENSION_COUNT']
for col in mora_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0)

print('[Target 구성 분석]')
print(f'전체 Y=1: {df_raw["TARGET_Y"].sum()}건')

# 하위 그룹 분석
y1_mask = df_raw['TARGET_Y'] == 1
if 'MORATORIUM_OVERDUE_AMOUNT' in df_raw.columns:
    has_mora_overdue = (df_raw['MORATORIUM_OVERDUE_AMOUNT'] > 0) & y1_mask
    has_suspension = (df_raw['ACCOUNT_SUSPENSION_COUNT'] > 0) & y1_mask
    print(f'  - MORATORIUM_OVERDUE > 0: {has_mora_overdue.sum()}건')
    print(f'  - ACCOUNT_SUSPENSION > 0: {has_suspension.sum()}건')

In [ ]:
# ============================================================
# E0-2: 다양한 Target 정의별 5-Fold CV 비교
# ============================================================

rskf_5 = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
results_e02 = []

# Target A: 현재 정의 (57건)
y_current = df['TARGET_Y']
if y_current.sum() >= 10:  # 최소 10건은 있어야 CV 가능
    aucs_a = cross_val_score(rf_v49, X, y_current, cv=rskf_5, scoring='roc_auc', n_jobs=-1)
    results_e02.append({
        'Target': f'A: 현재 정의 (n={y_current.sum()})',
        'AUC_mean': aucs_a.mean(), 'AUC_std': aucs_a.std(),
        'AUC_min': aucs_a.min(), 'AUC_max': aucs_a.max()
    })
    print(f'Target A (현재, n={y_current.sum()}): AUC={aucs_a.mean():.4f} +/- {aucs_a.std():.4f}')

# Target B: 엄격 정의 (MORATORIUM_OVERDUE > 0만)
if 'MORATORIUM_OVERDUE_AMOUNT' in df_raw.columns:
    y_strict = ((df_raw['MORATORIUM_OVERDUE_AMOUNT'] > 0) | 
                (df_raw['ACCOUNT_SUSPENSION_COUNT'] > 0)).astype(int)
    if y_strict.sum() >= 10:
        aucs_b = cross_val_score(rf_v49, X, y_strict, cv=rskf_5, scoring='roc_auc', n_jobs=-1)
        results_e02.append({
            'Target': f'B: 엄격 (n={y_strict.sum()})',
            'AUC_mean': aucs_b.mean(), 'AUC_std': aucs_b.std(),
            'AUC_min': aucs_b.min(), 'AUC_max': aucs_b.max()
        })
        print(f'Target B (엄격, n={y_strict.sum()}): AUC={aucs_b.mean():.4f} +/- {aucs_b.std():.4f}')

# Target C: 확대 정의 (MORATORIUM_COUNT > 0까지 포함)
if 'MORATORIUM_COUNT' in df_raw.columns:
    y_broad = ((df_raw['MORATORIUM_COUNT'] > 0) | (df_raw['TARGET_Y'] == 1)).astype(int)
    if y_broad.sum() >= 10:
        aucs_c = cross_val_score(rf_v49, X, y_broad, cv=rskf_5, scoring='roc_auc', n_jobs=-1)
        results_e02.append({
            'Target': f'C: 확대 (n={y_broad.sum()})',
            'AUC_mean': aucs_c.mean(), 'AUC_std': aucs_c.std(),
            'AUC_min': aucs_c.min(), 'AUC_max': aucs_c.max()
        })
        print(f'Target C (확대, n={y_broad.sum()}): AUC={aucs_c.mean():.4f} +/- {aucs_c.std():.4f}')

# 결과 정리
print('\n' + '=' * 70)
print('[E0-2 결과] Target 정의별 성능 비교')
print('=' * 70)
df_results = pd.DataFrame(results_e02)
print(df_results.to_string(index=False))

---
## 실험 E0-3: 추가 평가 지표 (PR-AUC, KS, Lift)

### Devil's Advocate 질문
> "AUC가 0.73이면 실제 운용 시 상위 10% 중 부실 기업을 몇 개나 잡는가?"  
> "불균형 데이터에서 ROC-AUC는 과대평가되는 경향이 있다"

In [ ]:
# ============================================================
# E0-3: 다면 평가 (PR-AUC, KS, Decile Lift)
# ============================================================
from sklearn.metrics import roc_curve

# 기존 모델의 test set 예측 사용
y_prob = rf_v49.predict_proba(X_test)[:, 1]

# 1. ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)

# 2. PR-AUC (불균형 데이터의 진짜 성능)
pr_auc = average_precision_score(y_test, y_prob)

# 3. KS Statistic (금융권 표준)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
ks_stat = np.max(tpr - fpr)

# 4. Brier Score
brier = brier_score_loss(y_test, y_prob)

# 5. Top-10% Lift
n_top = max(1, int(len(y_test) * 0.1))
top_idx = np.argsort(y_prob)[::-1][:n_top]
top_hit_rate = y_test.iloc[top_idx].mean()
base_rate = y_test.mean()
lift_10 = top_hit_rate / base_rate if base_rate > 0 else 0

print('=' * 60)
print('[E0-3 결과] 다면 평가 지표')
print('=' * 60)
print(f'ROC-AUC:        {roc_auc:.4f}')
print(f'PR-AUC:         {pr_auc:.4f}  (random baseline: {base_rate:.4f})')
print(f'KS Statistic:   {ks_stat:.4f}  (>0.3이면 금융권 실전 투입 가능)')
print(f'Brier Score:    {brier:.4f}  (낮을수록 좋음)')
print(f'Top-10% Lift:   {lift_10:.2f}x  (>5x이면 실용적)')
print(f'Top-10% 부실률:  {top_hit_rate*100:.1f}%  (전체 {base_rate*100:.1f}%)')
print('=' * 60)

# 판정
print('\n[Gate 0 판정]')
if ks_stat >= 0.3:
    print('  KS >= 0.3 --> 기본 판별력 확보. Phase 1 진행 가능.')
else:
    print(f'  KS = {ks_stat:.4f} < 0.3 --> 판별력 부족. Target 재정의 검토 필요.')

In [ ]:
# ============================================================
# E0-3 시각화: Precision-Recall Curve + KS Plot
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. PR Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
axes[0].plot(recall, precision, color='darkorange', linewidth=2)
axes[0].axhline(base_rate, color='gray', linestyle='--', label=f'Random: {base_rate:.3f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'PR Curve (PR-AUC={pr_auc:.4f})')
axes[0].legend()

# 2. KS Plot
axes[1].plot(thresholds, tpr[:-1], label='TPR', color='blue')
axes[1].plot(thresholds, fpr[:-1], label='FPR', color='red')
ks_threshold = thresholds[np.argmax(tpr[:-1] - fpr[:-1])]
axes[1].axvline(ks_threshold, color='green', linestyle='--', label=f'KS={ks_stat:.3f} @ {ks_threshold:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Rate')
axes[1].set_title('KS Statistic')
axes[1].legend()
axes[1].invert_xaxis()

# 3. Decile Lift Chart
n_deciles = 10
sorted_idx = np.argsort(y_prob)[::-1]
decile_size = len(y_test) // n_deciles
lifts = []
for i in range(n_deciles):
    start = i * decile_size
    end = start + decile_size if i < n_deciles - 1 else len(y_test)
    decile_idx = sorted_idx[start:end]
    decile_rate = y_test.iloc[decile_idx].mean()
    lifts.append(decile_rate / base_rate if base_rate > 0 else 0)

axes[2].bar(range(1, n_deciles+1), lifts, color='steelblue', edgecolor='black')
axes[2].axhline(1.0, color='red', linestyle='--', label='Baseline (Lift=1)')
axes[2].set_xlabel('Decile (1=highest risk)')
axes[2].set_ylabel('Lift')
axes[2].set_title('Decile Lift Chart')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Phase 0 결론 및 Gate 판정

### 기록 양식

| 측정 항목 | 값 | 판정 기준 | 통과 여부 |
|-----------|------|----------|----------|
| CV AUC Mean | ? | >= 0.70 | ? |
| CV AUC Std | ? | <= 0.05 | ? |
| KS Statistic | ? | >= 0.30 | ? |
| PR-AUC | ? | >= 0.15 | ? |
| Top-10% Lift | ? | >= 3.0x | ? |

### 다음 단계
- **모든 기준 통과** → Phase 1 (전처리 챌린지) 진행
- **KS < 0.30** → Target 재정의 우선 (E0-2 결과 기반)
- **AUC Std > 0.05** → 데이터 자체의 한계. 앙상블로 안정화 시도

In [ ]:
# ============================================================
# Phase 0 최종 요약
# ============================================================
print('\n' + '=' * 60)
print('PHASE 0: BASELINE CHALLENGE - FINAL SUMMARY')
print('=' * 60)
print(f'''
[기존 보고된 성능]
  AUC (single split, rs=42): {single_auc:.4f}

[실제 측정된 성능 (5-Fold CV x 3)]
  AUC Mean:  {cv_aucs.mean():.4f}
  AUC Std:   {cv_aucs.std():.4f}
  AUC 95%CI: [{cv_aucs.mean() - 1.96*cv_aucs.std():.4f}, {cv_aucs.mean() + 1.96*cv_aucs.std():.4f}]

[추가 지표]
  PR-AUC:      {pr_auc:.4f}
  KS Stat:     {ks_stat:.4f}
  Top-10% Lift: {lift_10:.2f}x

[Devil\'s Advocate 판정]
  0.7347이 진짜인가? --> CV Mean과의 차이: {single_auc - cv_aucs.mean():+.4f}
  결과가 안정적인가? --> Std: {cv_aucs.std():.4f} ({"안정" if cv_aucs.std() < 0.05 else "불안정"})
''')
print('=' * 60)
print('\n다음 단계: Phase 1 전처리 챌린지 노트북으로 이동')